In [ ]:
%%bash
exec_annotation -p tmp/profiles -k tmp/ko_list -o tmp/kofam.txt tmp/seq.fa --cpu 32 -f detail-tsv

In [ ]:
%%bash
diamond blastp \
    -q tmp/seq.fa \
    -d tmp/seq.fa \
    --out tmp/hit_seq.txt \
    --outfmt 6 qseqid sseqid nident qlen slen pident qcovhsp scovhsp bitscore evalue \
    --id 90 --subject-cover 95 --query-cover 95 \
    -k 0 --threads 32 --no-self-hits --masking 0 --quiet

In [ ]:
import pandas as pd
from Bio import SeqIO

aset = set()
for file in ['reference/reference.fasta', 'reference/AMRProt.fa']:
    with open(file) as handle:
        for record in SeqIO.parse(handle, 'fasta'):
            record.seq = record.seq[:-1] if record.seq[-1] == '*' else record.seq
            aset.add(record.seq)

with open('tmp/seq2description.fa', 'w') as output_handle:
    for file in ['env_nr', 'nr']:
        with open(f'tmp/{file}.fa') as handle:
            for title, seq in SeqIO.FastaIO.SimpleFastaParser(handle):
                if seq in aset:
                    output_handle.write('>%s\n%s\n' % (title, seq))

In [ ]:
seq2description = dict()
with open('tmp/seq2description.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        seq2description[record.seq] = record.description.split(' >')[0]

seq2source = dict()
with open('tmp/seq2source.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        seq2source[record.seq] = record.id

qset = set()
with open('tmp/hit_seq.txt') as f:
    for line in f:
        ls = line.rstrip().split('\t')
        if ls[0].split('|')[2] != ls[1].split('|')[2] and round(float(ls[5])) >= 95:
            if '*' in ls[0]:
                qset.add(ls[0])

ko = []
with open('tmp/kofam.txt') as f:
    for line in f:
        if line[0] != '#':
            ls = line.rstrip().split('\t')
            if ls[3] != '':
                ratio = float(ls[4]) / float(ls[3])
                ko.append((ls[1], ratio, ls[2] + ls[0] + '|' + ls[-1].split('"')[1].split(' [')[0]))
ko = pd.DataFrame(ko, columns = ['source', 'ratio', 'ko']).sort_values('ratio', ascending=False).groupby('source')['ko'].first().to_dict()

records = []
lines = []
with open('tmp/seq.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        if record.id not in qset:
            desc = seq2description.get(record.seq, 'NA')
            if not desc:
                continue

            line = [record.id, ko.get(record.id), seq2source.get(record.seq), desc.split(' ', 1)[-1].split(' [')[0].split('MULTISPECIES: ')[-1]]
            record.id = '|'.join(record.id.split('|')[:3] + [desc.split(' ', 1)[0]])
            record.description = desc.split(' ', 1)[-1]
            records.append(record)
    
            lines.append(line + [record.id])

with open('sarg_ref.fa', 'w') as output_handle:
    SeqIO.write(records, output_handle, 'fasta')

summary = pd.DataFrame(lines, columns = ['id', 'ko', 'source', 'description', 'sarg'])
summary['type'] = summary['id'].str.split('|').str.get(1)
summary['subtype'] = summary['id'].str.split('|').str.get(2)
summary['accession'] = summary['id'].str.split('|').str.get(3)
summary['definition'] = summary.ko.str.split('|').str.get(1)
summary['ko'] = summary.ko.str.split('|').str.get(0)

cols = ['type', 'subtype', 'sarg', 'source', 'description', 'ko', 'definition']
summary[cols].sort_values(cols).to_csv('misc/summary.tsv', index=False, sep='\t')